# Práctica guiada — Inferencia y contratos

Este notebook es para el alumnado. Completa las celdas marcadas con `TODO` y responde las preguntas antes de mirar la solución docente.

**Objetivo:** pasar de una fila CSV a una especificación clara para el módulo de inferencia de la clase 2.

## Cómo trabajar

1. Copia esta carpeta como tu espacio de trabajo.
2. Ejecuta `uv sync` desde `problem/starter/` cuando empiece la clase 2.
3. En esta práctica no implementes todavía el CLI completo.
4. Trabaja en parejas y cambia quién escribe cada diez minutos.
5. No abras `solutions/` hasta el debrief.

In [1]:
from dataclasses import dataclass
import csv
from pathlib import Path

from pydantic import BaseModel, ConfigDict, Field, ValidationError

DATA_PATH = Path("../../../../../../assets/03-wine-quality/inference_samples.csv")
print(DATA_PATH)
print(DATA_PATH.exists())

..\..\..\..\..\..\assets\03-wine-quality\inference_samples.csv
True


## 1. Leer los datos nuevos

### Práctica A — Inspección del CSV

Completa la celda siguiente para:

- Leer el CSV con `csv.DictReader`.
- Imprimir el número de filas.
- Imprimir las columnas.
- Guardar la primera fila en una variable llamada `row`.

In [2]:
# Lectura del CSV con csv.DictReader
with DATA_PATH.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    rows = list(reader)

row = rows[0] if rows else {}
print(f"Filas leidas: {len(rows)}")
print(f"Columnas: {fieldnames}")
print("Primera fila:", row)


Filas leidas: 5
Columnas: ['sample_id', 'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']
Primera fila: {'sample_id': 'red-001', 'fixed_acidity': '7.4', 'volatile_acidity': '0.7', 'citric_acid': '0.0', 'residual_sugar': '1.9', 'chlorides': '0.076', 'free_sulfur_dioxide': '11.0', 'total_sulfur_dioxide': '34.0', 'density': '0.9978', 'ph': '3.51', 'sulphates': '0.56', 'alcohol': '9.4'}


### Preguntas

1. ¿Qué campo identifica la muestra pero no es una característica del modelo?
2. ¿Cuántas características numéricas hay?
3. ¿Qué columna extra añadirías para probar el contrato?
4. ¿Qué valor de `ph` sería claramente inválido?

## 2. Construir el contrato con Pydantic

El contrato del modelo contiene estas características:

| Campo | Rango orientativo |
|---|---|
| `fixed_acidity` | 0–20 |
| `volatile_acidity` | 0–2 |
| `citric_acid` | 0–2 |
| `residual_sugar` | 0–20 |
| `chlorides` | 0–1 |
| `free_sulfur_dioxide` | 0–100 |
| `total_sulfur_dioxide` | 0–300 |
| `density` | 0.98–1.01 |
| `ph` | 2.5–4.5 |
| `sulphates` | 0–3 |
| `alcohol` | 5–20 |

In [3]:
class WineQualityRequestPydantic(BaseModel):
    """Mediciones fisico-quimicas de una muestra de vino tinto."""
    model_config = ConfigDict(extra="forbid")

    fixed_acidity: float = Field(ge=0, le=20)
    volatile_acidity: float = Field(ge=0, le=2)
    citric_acid: float = Field(ge=0, le=2)
    residual_sugar: float = Field(ge=0, le=20)
    chlorides: float = Field(ge=0, le=1)
    free_sulfur_dioxide: float = Field(ge=0, le=100)
    total_sulfur_dioxide: float = Field(ge=0, le=300)
    density: float = Field(ge=0.98, le=1.01)
    ph: float = Field(ge=2.5, le=4.5)
    sulphates: float = Field(ge=0, le=3)
    alcohol: float = Field(ge=5, le=20)

# Separa sample_id de la fila y valida solo las caracteristicas del modelo
model_row = {k: v for k, v in row.items() if k != "sample_id"}
request = WineQualityRequestPydantic.model_validate(model_row)
print("Validacion exitosa de la primera muestra:")
print(request.model_dump())


Validacion exitosa de la primera muestra:
{'fixed_acidity': 7.4, 'volatile_acidity': 0.7, 'citric_acid': 0.0, 'residual_sugar': 1.9, 'chlorides': 0.076, 'free_sulfur_dioxide': 11.0, 'total_sulfur_dioxide': 34.0, 'density': 0.9978, 'ph': 3.51, 'sulphates': 0.56, 'alcohol': 9.4}


### Práctica B — Validación

Completa una prueba para cada caso:

- Una fila válida.
- Una columna `unexpected_field`.
- Un `ph` fuera de rango.

Antes de ejecutar, escribe qué error esperas.

In [4]:
test_cases = {
    "Fila valida": model_row,
    "Columna unexpected_field": {**model_row, "unexpected_field": "not_allowed"},
    "pH fuera de rango (99)": {**model_row, "ph": 99.0},
    "Tipo no numerico en alcohol": {**model_row, "alcohol": "muy_fuerte"},
}

for name, test_payload in test_cases.items():
    try:
        WineQualityRequestPydantic.model_validate(test_payload)
        print(f"OK {name}: Validado correctamente")
    except ValidationError as error:
        field = error.errors()[0]["loc"][0]
        msg = error.errors()[0]["msg"]
        print(f"FALLO ESPERADO {name} -> Campo: '{field}' | Error: {msg}")


OK Fila valida: Validado correctamente
FALLO ESPERADO Columna unexpected_field -> Campo: 'unexpected_field' | Error: Extra inputs are not permitted
FALLO ESPERADO pH fuera de rango (99) -> Campo: 'ph' | Error: Input should be less than or equal to 4.5
FALLO ESPERADO Tipo no numerico en alcohol -> Campo: 'alcohol' | Error: Input should be a valid number, unable to parse string as a number


## 3. Comparar con `dataclass`

Construye la misma estructura con `dataclass`. No añadas todavía validación manual: observa qué garantiza Python y qué no garantiza.

In [5]:
@dataclass(frozen=True, slots=True)
class WineQualityRequestDataclass:
    fixed_acidity: float
    volatile_acidity: float
    citric_acid: float
    residual_sugar: float
    chlorides: float
    free_sulfur_dioxide: float
    total_sulfur_dioxide: float
    density: float
    ph: float
    sulphates: float
    alcohol: float

# Instancia valida
valid_dc = WineQualityRequestDataclass(**{k: float(v) for k, v in model_row.items()})
print("Dataclass valido creado:", valid_dc)

# Probamos que ocurre con ph=99.0:
invalid_ph_payload = {k: float(v) for k, v in model_row.items()}
invalid_ph_payload["ph"] = 99.0
dc_with_bad_ph = WineQualityRequestDataclass(**invalid_ph_payload)
print("Dataclass con ph=99.0 creado SIN error (no valida rangos):", dc_with_bad_ph.ph)


Dataclass valido creado: WineQualityRequestDataclass(fixed_acidity=7.4, volatile_acidity=0.7, citric_acid=0.0, residual_sugar=1.9, chlorides=0.076, free_sulfur_dioxide=11.0, total_sulfur_dioxide=34.0, density=0.9978, ph=3.51, sulphates=0.56, alcohol=9.4)
Dataclass con ph=99.0 creado SIN error (no valida rangos): 99.0


### Decisión de diseño

> **Usaría Pydantic en** la frontera externa del sistema (lectura de CSV, APIs HTTP y validación de entrada/salida) **porque** realiza parseo automático de tipos desde strings, rechaza campos extra no autorizados (`extra="forbid"`), comprueba límites y rangos de negocio (`Field(ge=..., le=...)`) y proporciona mensajes de error estructurados (`ValidationError`).
>
> **Usaría `dataclass` en** las capas internas de preprocesado e inferencia (`WineFeatures`) **porque** los datos ya han sido validados previamente en la frontera, no se necesita el sobrecoste (*overhead*) de revalidar en cada paso y ofrece una estructura ligera, inmutable (`frozen=True`) y rápida en memoria para garantizar el orden exacto del vector numérico que consume el estimador.


## 4. Definir el orden del vector

El modelo no recibe un diccionario. Recibe una lista numérica con un orden estable.

In [6]:
FEATURE_NAMES = (
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
)

vector = [getattr(request, name) for name in FEATURE_NAMES]
print("Vector numerico en el orden del modelo:")
print(vector)
assert len(vector) == 11


Vector numerico en el orden del modelo:
[7.4, 0.7, 0.0, 1.9, 0.076, 11.0, 34.0, 0.9978, 3.51, 0.56, 9.4]


### Práctica C — Error de orden

Intercambia mentalmente `density` y `alcohol`. Responde:

- ¿El vector sigue teniendo 11 valores?
- ¿El clasificador detectaría necesariamente el error?
- ¿Cómo evitarías que cada script definiera un orden distinto?

## 5. Contrato de salida

Diseña la estructura que devolverá el módulo. Debe permitir responder:

- ¿Qué muestra se ha procesado?
- ¿Qué categoría se ha predicho?
- ¿Con qué confianza?
- ¿Qué versión de modelo y preprocesado se utilizaron?

In [7]:
from typing import Literal

QualityBand = Literal["needs_review", "acceptable", "excellent"]

class WineQualityPredictionPydantic(BaseModel):
    """Prediccion de categoria y metadatos del artefacto que la genero."""
    quality_band: QualityBand
    confidence: float = Field(ge=0, le=1)
    model_version: str
    preprocessing_version: str

# Ejemplo de salida para red-001:
prediction_sample = WineQualityPredictionPydantic(
    quality_band="acceptable",
    confidence=0.85,
    model_version="wine-quality-rf-v1",
    preprocessing_version="wine-red-features-v1",
)
full_output = {
    "sample_id": "red-001",
    **prediction_sample.model_dump()
}
print("Salida completa para red-001:")
print(full_output)


Salida completa para red-001:
{'sample_id': 'red-001', 'quality_band': 'acceptable', 'confidence': 0.85, 'model_version': 'wine-quality-rf-v1', 'preprocessing_version': 'wine-red-features-v1'}


## 6. Demo del docente: observar la inferencia

El docente ejecutará el notebook resuelto y el comando de referencia. Mientras tanto, anota el recorrido:

```text
CSV → contrato → preprocesado → modelo cargado → predicción
```

No copies la implementación. Tu código de producción empieza en la clase 2 con el starter.

## Entrega

Entrega una captura o exportación con:

- La tabla de contrato.
- Las respuestas de las prácticas A, B y C.
- El orden de las 11 características.
- Un ejemplo de entrada inválida.
- Un contrato de salida.
- Tu plan de implementación para `contracts.py`, `preprocess.py`, `inference.py` y `predict_file.py`.

# 📋 Resumen de Entrega — Assignment 3.1

### 1. Tabla de contrato (Entrada)

| Campo | Tipo | Unidad | Rango razonable | Nulos | Justificación |
|---|---|---|---|---|---|
| `fixed_acidity` | `float` | g(ácido tartárico)/dm³ | [0.0, 20.0] | No | Acidez fija presente en vinos tintos. |
| `volatile_acidity` | `float` | g(ácido acético)/dm³ | [0.0, 2.0] | No | Niveles excesivos causan acescencia (olor a vinagre). |
| `citric_acid` | `float` | g/dm³ | [0.0, 2.0] | No | Aporta frescura; presente en bajas proporciones. |
| `residual_sugar` | `float` | g/dm³ | [0.0, 20.0] | No | Azúcar remanente tras la fermentación alcohólica. |
| `chlorides` | `float` | g(NaCl)/dm³ | [0.0, 1.0] | No | Contenido mineral y salinidad del vino. |
| `free_sulfur_dioxide` | `float` | mg/dm³ | [0.0, 100.0] | No | SO₂ libre activo contra oxidación microbiana. |
| `total_sulfur_dioxide` | `float` | mg/dm³ | [0.0, 300.0] | No | Suma de azufre libre y ligado (límite legal enológico). |
| `density` | `float` | g/cm³ | [0.98, 1.01] | No | Cercano a la densidad del agua según azúcar y alcohol. |
| `ph` | `float` | Escala pH | [2.5, 4.5] | No | Rango fisiológico y de estabilidad microbiológica. |
| `sulphates` | `float` | g(K₂SO₄)/dm³ | [0.0, 3.0] | No | Aditivo antioxidante y conservante. |
| `alcohol` | `float` | % vol | [5.0, 20.0] | No | Graduación alcohólica típica de fermentación vínica. |

> *Nota:* `sample_id` (str) identifica la muestra en la frontera de negocio pero **se excluye estrictamente** del vector numérico del modelo.

---

### 2. Respuestas a las Prácticas

#### Práctica A — Inspección del CSV
1. **¿Qué campo identifica la muestra pero no es una característica del modelo?**
   - El campo `sample_id` (ej. `red-001`).
2. **¿Cuántas características numéricas hay?**
   - Exactamente **11** características numéricas de tipo continuo (`float`).
3. **¿Qué columna extra añadirías para probar el contrato?**
   - Cualquier campo espurio, como `"unexpected_field": "not_allowed"` o `"quality": 6`.
4. **¿Qué valor de `ph` sería claramente inválido?**
   - Un valor de `99.0` (físicamente absurdo para soluciones acuosas de vino) o un valor negativo/alcalino como `8.5`.

#### Práctica B — Validación
- **Fila válida:** Se valida sin excepciones retornando el objeto `WineQualityRequestPydantic`.
- **Columna `unexpected_field`:** Pydantic lanza `ValidationError` con `extra_forbidden` porque `model_config = ConfigDict(extra="forbid")`.
- **`ph=99.0`:** Lanza `ValidationError` con `less_than_equal` al superar el límite fijado en `Field(le=4.5)`.
- **Tipo incorrecto (`alcohol="muy_fuerte"`):** Lanza `ValidationError` con `float_parsing`.

#### Práctica C — Error de orden
1. **¿El vector sigue teniendo 11 valores si se intercambian `density` y `alcohol`?**
   - Sí, la longitud y tipos son idénticos (`list[float]` de dimensión 11).
2. **¿El clasificador detectaría necesariamente el error?**
   - **No.** scikit-learn recibirá una matriz $(1, 11)$ válida y producirá una predicción sin lanzar excepciones, pero el resultado será silenciosamente erróneo al evaluar reglas de densidad sobre porcentaje de alcohol y viceversa.
3. **¿Cómo evitarías que cada script definiera un orden distinto?**
   - Centralizando la tupla canónica `FEATURE_NAMES` en un módulo único (`preprocess.py`) y verificando en la carga del artefacto que los metadatos `feature_names` coincidan exactamente antes de ejecutar inferencias.

---

### 3. Orden canónico de las 11 características

El vector numérico de entrada sigue estrictamente este orden:
```python
FEATURE_NAMES = (
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
)
```

---

### 4. Ejemplo de entrada inválida y captura de error

```python
invalid_sample = {
    "fixed_acidity": 7.4,
    "volatile_acidity": 0.7,
    "citric_acid": 0.0,
    "residual_sugar": 1.9,
    "chlorides": 0.076,
    "free_sulfur_dioxide": 11.0,
    "total_sulfur_dioxide": 34.0,
    "density": 0.9978,
    "ph": 99.0,  # <-- Inválido (debe ser <= 4.5)
    "sulphates": 0.56,
    "alcohol": 9.4,
    "unexpected_field": "no_permitido",  # <-- Inválido (extra="forbid")
}
```
**Etapa de rechazo:** Inmediatamente en la frontera de entrada (`contracts.py`), arrojando `ValidationError` antes de tocar el modelo.

---

### 5. Contrato de salida

```python
class WineQualityPrediction(BaseModel):
    quality_band: Literal["needs_review", "acceptable", "excellent"]
    confidence: float = Field(ge=0.0, le=1.0)
    model_version: str
    preprocessing_version: str
```

**Salida generada para `red-001`:**
```json
{
    "sample_id": "red-001",
    "quality_band": "acceptable",
    "confidence": 0.85,
    "model_version": "wine-quality-rf-v1",
    "preprocessing_version": "wine-red-features-v1"
}
```

---

### 6. Plan de implementación para la Clase 2 (Módulo local)

1. **`contracts.py`**: Modelos Pydantic `WineQualityRequest` (con `extra="forbid"` y rangos enológicos) y `WineQualityPrediction` (con las bandas válidas de calidad).
2. **`preprocess.py`**: Constante `FEATURE_NAMES` inmutable, dataclass `WineFeatures` (`frozen=True`) y función `preprocess_wine_request()` para ordenar deterministamente el vector numérico sin `sample_id`.
3. **`inference.py`**: `load_wine_quality_model()` (verificando estructura del artefacto y coincidencia exacta de nombres) y función `infer_wine_quality()` calculando categoría y probabilidad máxima.
4. **`predict_file.py`**: CLI con `argparse` que lee el CSV, valida cada fila y acumula las predicciones en memoria, garantizando **atomicidad** (si una fila es inválida se aborta sin dejar salida parcial en disco).
